# Make_Dyna1 — Batch Dyna-1 predictions for DynaFeatureDock

Runs **Dyna-1 (ESM-2 version)** on all FeatureDock PDB structures and saves per-residue flexibility CSVs to Google Drive.

**Workflow:**
1. Mount Google Drive
2. Upload your `PDB_labled/` folder to Drive (or zip and upload here)
3. Install dependencies & download Dyna-1 weights (no gated model needed)
4. Run Dyna-1 on every PDB → saves `{pdbid}-Dyna1.csv` to `Drive/dyna1_csvs/`
5. Back in your local `DynaFeatureDock_Pipeline.ipynb`, point `DYNA1_CSV_DIR` to the downloaded folder

**Note:** This uses the **ESM-2 version** of Dyna-1 — no HuggingFace gated-model access required.
The ESM-3 version gives slightly better predictions but requires ESM-3 access approval.
For ~4500 structures on a Colab T4 GPU, expect ~3-5 hours.

## Step 0 — IMPORTANT: Fix numpy first (Colab-specific bug)
Run this cell alone **before** doing Runtime → Run all. The session will restart — that is normal.

In [ ]:
import os, signal
import numpy

if numpy.__version__ != '1.26.4':
    print(f'numpy {numpy.__version__} → installing 1.26.4 and restarting...')
    os.system('pip uninstall -y numpy')
    os.system('pip install numpy==1.26.4')
    os.kill(os.getpid(), signal.SIGKILL)   # Colab will restart kernel
else:
    print(f'numpy {numpy.__version__} OK — continue to next cell')

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
# Output folder on Drive — CSVs will be saved here
DRIVE_CSV_DIR = '/content/drive/MyDrive/dyna1_csvs'
os.makedirs(DRIVE_CSV_DIR, exist_ok=True)
print(f'CSV output: {DRIVE_CSV_DIR}')

## Step 2 — Upload PDB files

You need your `PDB_labled/` folder (4507 PDB files) in Colab.
**Choose one option:**

**Option A (recommended):** Upload a zip of `PDB_labled/` to your Drive, then run:
```python
import zipfile, shutil
with zipfile.ZipFile('/content/drive/MyDrive/PDB_labled.zip', 'r') as z:
    z.extractall('/content/PDB_labled')
```

**Option B:** If PDB_labled is already on your Drive:
```python
PDB_DIR = '/content/drive/MyDrive/PDB_labled'
```

**Option C:** Upload directly from local machine (slow for 4500 files — use a zip):
```python
from google.colab import files
uploaded = files.upload()   # select PDB_labled.zip
```

After uploading, set `PDB_DIR` below and run this cell.

In [ ]:
import os, glob

# ── SET THIS to where your PDB files are ────────────────────────────────────
PDB_DIR = '/content/PDB_labled'   # change if using Drive path
# ────────────────────────────────────────────────────────────────────────────

# Option A: unzip from Drive
ZIP_ON_DRIVE = '/content/drive/MyDrive/PDB_labled.zip'
if not os.path.isdir(PDB_DIR) and os.path.isfile(ZIP_ON_DRIVE):
    print('Unzipping PDB_labled.zip from Drive...')
    import zipfile
    os.makedirs(PDB_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_ON_DRIVE, 'r') as z:
        z.extractall(PDB_DIR)
    print('Done.')

pdb_files = sorted(glob.glob(os.path.join(PDB_DIR, '**', '*.pdb'), recursive=True) +
                   glob.glob(os.path.join(PDB_DIR, '*.pdb')))
print(f'Found {len(pdb_files)} PDB files in {PDB_DIR}')
if pdb_files:
    print('Sample:', pdb_files[:3])

## Step 3 — Install dependencies & clone Dyna-1

In [ ]:
import os

# Clone Dyna-1 repo
if not os.path.isdir('/content/Dyna-1'):
    os.system('git clone https://github.com/WaymentSteeleLab/Dyna-1.git --depth 1')
    print('Dyna-1 repo cloned.')
else:
    print('Dyna-1 repo already present.')

# Install deps from requirements.txt  (torch is already in Colab)
print('Installing Dyna-1 requirements...')
os.system('pip install -q -r /content/Dyna-1/requirements.txt')
os.system('pip install -q torcheval mdtraj MDAnalysis biopython')
print('Done.')

## Step 4 — Download Dyna-1 weights

Downloads **`dyna1-esm2.pt`** (ESM-2 version, ~200 MB) from HuggingFace — no token required.

In [ ]:
import os

WEIGHTS_DIR  = '/content/Dyna-1/model/weights'
WEIGHTS_FILE = os.path.join(WEIGHTS_DIR, 'dyna1-esm2.pt')
os.makedirs(WEIGHTS_DIR, exist_ok=True)

if not os.path.isfile(WEIGHTS_FILE):
    print('Downloading dyna1-esm2.pt from HuggingFace (no login required)...')
    os.system('pip install -q huggingface_hub')
    os.system(f'huggingface-cli download gelnesr/Dyna-1 dyna1-esm2.pt --local-dir "{WEIGHTS_DIR}"')
    print('Done.')
else:
    print(f'Weights already at {WEIGHTS_FILE}')

assert os.path.isfile(WEIGHTS_FILE), f'ERROR: weights not found at {WEIGHTS_FILE}'

## Step 5 — Load Dyna-1 model

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')

# Add Dyna-1 to path so local imports work
DYNA1_ROOT = '/content/Dyna-1'
if DYNA1_ROOT not in sys.path:
    sys.path.insert(0, DYNA1_ROOT)

import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer
from model.model import ESM_model   # from the Dyna-1 repo
import utils                         # Dyna-1 utils (prob_adjusted)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# Load model
model = ESM_model(method='esm2', nheads=8, nlayers=12, layer=30).to(DEVICE)
state = torch.load('/content/Dyna-1/model/weights/dyna1-esm2.pt', map_location=DEVICE)
model.load_state_dict(state, strict=False)
model.eval()
print('Dyna-1 (ESM-2) model loaded.')

# Load tokenizer (small ESM-2 for tokenization only, ~30 MB)
tokenizer = AutoTokenizer.from_pretrained('facebook/esm2_t6_8M_UR50D')
print('Tokenizer loaded.')

## Step 6 — Extract sequences from PDB files

In [ ]:
import re

AA3TO1 = {
    'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E',
    'GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F',
    'PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V',
    # common non-standard → nearest standard
    'MSE':'M','SEC':'C','PYL':'K','HSD':'H','HSE':'H','HSP':'H',
    'HID':'H','HIE':'H','HIP':'H','CYX':'C','CYM':'C','LYN':'K',
}

def pdb_to_sequence(pdbfile, chain='A'):
    """Extract Cα sequence (deduplicated by resnum) from a PDB file."""
    residues = []  # (resnum, aa1)
    seen = set()
    with open(pdbfile) as f:
        for line in f:
            if not line.startswith('ATOM'):
                continue
            if line[21] != chain:
                continue
            if line[12:16].strip() != 'CA':
                continue
            resnum  = int(line[22:26])
            resname = line[17:20].strip()
            if resnum not in seen:
                seen.add(resnum)
                residues.append((resnum, AA3TO1.get(resname, 'X')))
    if not residues:
        return None, []
    resnums = [r[0] for r in residues]
    seq = ''.join(r[1] for r in residues)
    return seq, resnums

# Validate protein-only alphabet for Dyna-1
PROTEIN_RE = re.compile('^[ACDEFGHIKLMNPQRSTVWY]+$', re.I)

print('Sequence extraction function ready.')
# Quick test
if pdb_files:
    seq_test, rn_test = pdb_to_sequence(pdb_files[0])
    if seq_test:
        print(f'Test ({os.path.basename(pdb_files[0])}): {len(seq_test)} residues, seq[:20]={seq_test[:20]}')
    else:
        print('No chain A in test file — check chain ID.')

## Step 7 — Run Dyna-1 on all PDB files and save CSVs

Saves `{pdbid}-Dyna1.csv` (columns: `position`, `residue`, `p_exchange`) to `DRIVE_CSV_DIR`.

Already-completed structures are skipped automatically — safe to rerun if interrupted.

In [ ]:
import time

@torch.no_grad()
def run_dyna1_esm2(sequence):
    """Run Dyna-1 ESM-2 on a single sequence. Returns numpy array of p_exchange."""
    seq_input  = tokenizer.encode(sequence, add_special_tokens=False, return_tensors='pt').to(DEVICE)
    seq_mask   = seq_input != 0
    logits     = model(seq_input, seq_mask)
    p          = utils.prob_adjusted(logits).cpu().numpy().squeeze()
    return p


def process_pdb(pdbfile, out_dir, chain='A'):
    """
    Run Dyna-1 on one PDB file. Saves CSV to out_dir.
    Returns 'done', 'skipped' (already exists), or 'error:msg'.
    """
    pdbid    = os.path.splitext(os.path.basename(pdbfile))[0].lower()
    out_csv  = os.path.join(out_dir, f'{pdbid}-Dyna1.csv')

    if os.path.isfile(out_csv):
        return 'skipped'

    seq, resnums = pdb_to_sequence(pdbfile, chain=chain)
    if seq is None or len(seq) < 5:
        return f'error:no_chain_{chain}'
    seq_clean = ''.join(aa if PROTEIN_RE.match(aa) else 'G' for aa in seq)

    try:
        p = run_dyna1_esm2(seq_clean)
    except Exception as e:
        return f'error:{str(e)[:80]}'

    # Align output to actual residue numbers
    n = min(len(p), len(resnums))
    df = pd.DataFrame({
        'position':   np.arange(1, n + 1),
        'resnum':     resnums[:n],
        'residue':    list(seq[:n]),
        'p_exchange': p[:n],
    })
    df.to_csv(out_csv, index=False)
    return 'done'


# ── Main batch loop ──────────────────────────────────────────────────────────
n_done, n_skip, n_err = 0, 0, 0
errors = []
t0 = time.time()

for i, pdbfile in enumerate(pdb_files):
    status = process_pdb(pdbfile, DRIVE_CSV_DIR)

    if status == 'done':    n_done += 1
    elif status == 'skipped': n_skip += 1
    else:
        n_err += 1
        pdbid = os.path.splitext(os.path.basename(pdbfile))[0]
        errors.append((pdbid, status))

    if (i + 1) % 100 == 0:
        elapsed = time.time() - t0
        eta_min = (elapsed / (i + 1)) * (len(pdb_files) - i - 1) / 60
        print(f'[{i+1}/{len(pdb_files)}]  done={n_done}  skip={n_skip}  err={n_err}  ETA={eta_min:.0f} min')

elapsed_total = (time.time() - t0) / 60
print(f'\nFinished in {elapsed_total:.1f} min')
print(f'Done: {n_done}  |  Skipped (already existed): {n_skip}  |  Errors: {n_err}')

if errors:
    print('\nFirst 20 errors:')
    for pdbid, msg in errors[:20]:
        print(f'  {pdbid}: {msg}')

## Step 8 — Verify output

In [ ]:
import glob

csvs = glob.glob(os.path.join(DRIVE_CSV_DIR, '*-Dyna1.csv'))
print(f'Total CSV files in {DRIVE_CSV_DIR}: {len(csvs)}')

if csvs:
    sample = pd.read_csv(csvs[0])
    print(f'\nSample ({os.path.basename(csvs[0])}):')
    print(sample.head(10).to_string(index=False))
    print(f'\nMean p_exchange: {sample.p_exchange.mean():.3f}')
    print(f'Residues: {len(sample)}')

## Step 9 — Download the CSV folder

**Option A:** The CSVs are already saved to your Google Drive at `MyDrive/dyna1_csvs/`.
Download the whole folder from the Drive web UI (right-click → Download).

**Option B:** Zip and download directly from Colab:

In [ ]:
# Optional: zip all CSVs and download in one click
# Comment out if you prefer to download from Drive directly

from google.colab import files
import shutil

ZIP_OUT = '/content/dyna1_csvs.zip'
shutil.make_archive('/content/dyna1_csvs', 'zip', DRIVE_CSV_DIR)
print(f'Zip size: {os.path.getsize(ZIP_OUT) / 1e6:.1f} MB')
files.download(ZIP_OUT)

## Next steps — back in DynaFeatureDock_Pipeline.ipynb

1. Place the downloaded `dyna1_csvs/` folder somewhere accessible (e.g., inside `dyna_featuredock/`).
2. In **Cell-04 (Config)**, set:
   ```python
   DYNA1_CSV_DIR = str(PROJECT_ROOT / 'dyna_featuredock' / 'dyna1_csvs')
   DYNA_METHOD   = 'csv'   # use pre-computed CSVs instead of running Dyna-1
   ```
3. Cell-07 will automatically load from the CSV folder.

CSV format produced here: `position, resnum, residue, p_exchange`